In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Step 1: Data Load + Overview**

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('/kaggle/input/datasets/muhammadumer7804/zamato-dataset/zomato.csv')

# Check shape of the dataset (rows, columns)
print("Shape:", df.shape)

In [ ]:
# Display first 5 rows
df.head()

In [ ]:
# Column names, data types, and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Count missing values in each column
df.isnull().sum()

# **Step 2: Handle Missing Values**

In [ ]:
# Drop 'dish_liked' column since more than 50% values are missing
df.drop(columns=['dish_liked'], inplace=True)

In [ ]:
# Drop rows where important columns like location, rest_type, cuisines are missing
# (these have very few missing rows, safe to drop)
df.dropna(subset=['location', 'rest_type', 'cuisines'], inplace=True)

In [ ]:
# Fill missing 'rate' with a placeholder for now (we will clean the format in next step)
df['rate'] = df['rate'].fillna('NEW')

# Fill missing 'approx_cost' with the median (will convert to numeric in next step)
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].fillna(
    df['approx_cost(for two people)'].mode()[0]
)

In [ ]:
# Fill missing phone numbers with 'Not Available'
df['phone'] = df['phone'].fillna('Not Available')

In [ ]:
# Verify missing values after cleaning
df.isnull().sum()

# **Step 3: Duplicates Check & Remove**

In [ ]:
# Check how many duplicate rows exist
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Reset index after dropping rows
df.reset_index(drop=True, inplace=True)

In [ ]:
# Check new shape after removing duplicates
print("Shape after removing duplicates:", df.shape)

# **Step 4: Fix Data Types**

In [ ]:
# Replace 'NEW' and '-' with NaN first
df['rate'] = df['rate'].replace('NEW', np.nan)
df['rate'] = df['rate'].replace('-', np.nan)

# Remove '/5' part from rate column (e.g. "4.1/5" -> "4.1")
df['rate'] = df['rate'].str.split('/').str[0]

# Convert rate column to float
df['rate'] = df['rate'].astype(float)

In [ ]:
# Fill missing rate values (from 'NEW'/'-') with the median rating
df['rate'] = df['rate'].fillna(df['rate'].median())

In [ ]:
# Remove commas (e.g. "1,200" -> "1200") and convert to integer
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '')
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(int)

In [ ]:
# Check updated data types
df.dtypes

In [ ]:
# Verify everything is still correct after restart
print(df.dtypes)
print(df.isnull().sum())

# **Step 5: Clean Text/Categorical Columns**

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

In [ ]:
# Rename columns to simpler names
df = df.rename(columns={
    'approx_cost(for two people)': 'approx_cost',
    'listed_in(type)': 'listed_type',
    'listed_in(city)': 'listed_city'
})

In [ ]:
# Remove extra spaces from restaurant names
df['name'] = df['name'].str.strip()

In [ ]:
# Check unique values
print(df['online_order'].unique())
print(df['book_table'].unique())

In [ ]:
# Remove extra spaces from location column
df['location'] = df['location'].str.strip()

# Check how many unique locations exist
print(df['location'].nunique())

In [ ]:
# Remove extra spaces from rest_type column
df['rest_type'] = df['rest_type'].str.strip()

# Check unique values
print(df['rest_type'].unique())

In [ ]:
# Remove extra spaces from cuisines column
df['cuisines'] = df['cuisines'].str.strip()

# Check how many unique cuisine combinations exist
print(df['cuisines'].nunique())

In [ ]:
# Remove extra spaces
df['listed_type'] = df['listed_type'].str.strip()
df['listed_city'] = df['listed_city'].str.strip()

# Check unique values
print(df['listed_type'].unique())

In [ ]:
# Replace line break characters in phone column with a comma
df['phone'] = df['phone'].str.replace('\r\n', ', ')

In [ ]:
# Check sample phone values
df['phone'].head()

# **Step 6: Outliers Check**

In [ ]:
import matplotlib.pyplot as plt

# Boxplot for votes column
plt.figure(figsize=(6,4))
plt.boxplot(df['votes'])
plt.title('Votes Column - Outlier Check')
plt.show()

In [ ]:
# Boxplot for approx_cost column
plt.figure(figsize=(6,4))
plt.boxplot(df['approx_cost'])
plt.title('Approx Cost Column - Outlier Check')
plt.show()

In [ ]:
# Boxplot for rate column
plt.figure(figsize=(6,4))
plt.boxplot(df['rate'])
plt.title('Rate Column - Outlier Check')
plt.show()

In [ ]:
# Check min, max and quartile values
df[['votes', 'approx_cost', 'rate']].describe()

# **Step 7: Save Final Cleaned Dataset**

In [ ]:
# Save the cleaned dataset as a new CSV file
df.to_csv('zomato_cleaned.csv', index=False)

# Confirm final shape
print("Final cleaned dataset shape:", df.shape)

In [ ]:
# View final cleaned dataset
df.head()

# **EDA Start**

# **EDA Step 1: Load Cleaned Dataset**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set(style="whitegrid")

In [ ]:
# Check if df already exists and is cleaned
print(df.shape)
df.head()

# **EDA Step 2: Univariate Analysis**

In [ ]:
# Distribution of restaurant ratings
plt.figure(figsize=(8,5))
sns.histplot(df['rate'], bins=20, kde=True)
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

In [ ]:
# Distribution of approx cost for two people
plt.figure(figsize=(8,5))
sns.histplot(df['approx_cost'], bins=30, kde=True)
plt.title('Distribution of Approx Cost (For Two People)')
plt.xlabel('Approx Cost')
plt.ylabel('Count')
plt.show()

In [ ]:
# Count of restaurants offering online order
plt.figure(figsize=(6,4))
sns.countplot(x='online_order', data=df)
plt.title('Online Order Availability')
plt.xlabel('Online Order')
plt.ylabel('Count')
plt.show()

In [ ]:
# Count of restaurants offering table booking
plt.figure(figsize=(6,4))
sns.countplot(x='book_table', data=df)
plt.title('Table Booking Availability')
plt.xlabel('Table Booking')
plt.ylabel('Count')
plt.show()

In [ ]:
# Top 10 most common restaurant types
plt.figure(figsize=(10,5))
df['rest_type'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 Restaurant Types')
plt.xlabel('Restaurant Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# **EDA Step 3: Cuisines Analysis**

In [ ]:
# Since cuisines column has multiple values per row, split and count individually
all_cuisines = df['cuisines'].str.split(', ')
cuisines_list = all_cuisines.explode()

# Count top 10 most common cuisines
top_cuisines = cuisines_list.value_counts().head(10)

plt.figure(figsize=(10,5))
top_cuisines.plot(kind='bar')
plt.title('Top 10 Most Common Cuisines')
plt.xlabel('Cuisine')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# **EDA Step 4: Location-wise Analysis**

In [ ]:
# Top 10 locations with most restaurants
plt.figure(figsize=(10,5))
df['location'].value_counts().head(10).plot(kind='bar')
plt.title('Top 10 Locations by Restaurant Count')
plt.xlabel('Location')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Get top 10 locations first
top_locations = df['location'].value_counts().head(10).index

# Filter dataframe for only these locations
df_top_locations = df[df['location'].isin(top_locations)]

# Average cost per location
avg_cost_location = df_top_locations.groupby('location')['approx_cost'].mean().sort_values(ascending=False)

plt.figure(figsize=(10,5))
avg_cost_location.plot(kind='bar')
plt.title('Average Cost by Location (Top 10 Locations)')
plt.xlabel('Location')
plt.ylabel('Average Cost')
plt.xticks(rotation=45)
plt.show()

# **EDA Step 5: Online Order & Table Booking Impact on Rating**

In [ ]:
# Average rating for restaurants with vs without online order
plt.figure(figsize=(6,4))
sns.boxplot(x='online_order', y='rate', data=df)
plt.title('Rating by Online Order Availability')
plt.xlabel('Online Order')
plt.ylabel('Rating')
plt.show()

In [ ]:
# Average rating for restaurants with vs without table booking
plt.figure(figsize=(6,4))
sns.boxplot(x='book_table', y='rate', data=df)
plt.title('Rating by Table Booking Availability')
plt.xlabel('Table Booking')
plt.ylabel('Rating')
plt.show()

In [ ]:
# Average cost comparison
print("Average cost (Online Order):")
print(df.groupby('online_order')['approx_cost'].mean())

print("\nAverage cost (Table Booking):")
print(df.groupby('book_table')['approx_cost'].mean())

# **EDA Step 6: Correlation Analysis**

In [ ]:
# Correlation matrix for numeric columns
numeric_cols = df[['rate', 'votes', 'approx_cost']]
correlation = numeric_cols.corr()

plt.figure(figsize=(6,5))
sns.heatmap(correlation, annot=True, cmap='coolwarm')
plt.title('Correlation Between Rate, Votes and Cost')
plt.show()

In [ ]:
# Relationship between votes and rating
plt.figure(figsize=(8,5))
sns.scatterplot(x='votes', y='rate', data=df, alpha=0.3)
plt.title('Votes vs Rating')
plt.xlabel('Votes')
plt.ylabel('Rating')
plt.show()